# Wan teacher dataset — shard 04 of 04
**10 planned candidates**, 50 steps, **480 × 832**, 81 frames, 16 fps. This uses the settings of your successful quality test, not the rejected 30-step version.

Use a T4 GPU or another authorized CUDA GPU. This is one portable work shard, not an account-switching workflow. Colab prohibits using multiple accounts to work around access/resource restrictions. Use an approved allocation and follow the service and institute rules.

Estimated budget based on your previous measurement: approximately **7.5 GPU-hours for ten videos**, plus setup/encoding/export. This is an estimate, not a runtime guarantee; prompts, hardware and load can change it. A session may end before the shard completes. Every finished clip is committed separately and verified on rerun. An unfinished clip restarts from the beginning.

Start by generating one candidate and inspect it. Do not generate nine more if the event is wrong. Detailed prompts improve specification but do not guarantee physics or prompt following. Forty generated candidates may yield fewer than forty accepted videos. This is a small pilot dataset, not evidence of general reverse-time learning.

The notebook installs its own **Python 3.11** and pinned GPU packages. The default Colab kernel can be newer. No old-project requirements, kernel replacement or package downgrades are needed.

## 1. Mount Drive and select this shard
Run from the top. Distinct shards have distinct IDs and output folders. Do not run two copies of the same shard simultaneously.

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
SHARD=4
WORK=Path('/content/wan_dataset_work'); WORK.mkdir(exist_ok=True)
ROOT=Path('/content/drive/MyDrive/reverse_video_wan_shards_v1')/f'shard_{SHARD:02d}'
ROOT.mkdir(parents=True,exist_ok=True)
EXPORTS=ROOT.parent/'transfer_bundles'; EXPORTS.mkdir(exist_ok=True)
ENV=os.environ.copy()
ENV.pop('PYTHONPATH',None); ENV.pop('PYTHONHOME',None)
ENV.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false',HF_HOME=str(WORK/'hf_cache'),PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
VENV=WORK/'gpu_env'; PY=VENV/'bin/python'

def stream(command,env=ENV):
    print('Running:', ' '.join(map(str,command)),flush=True)
    child=subprocess.Popen(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1,env=env)
    try:
        for line in child.stdout: print(line,end='',flush=True)
        rc=child.wait()
        if rc: raise subprocess.CalledProcessError(rc,command)
    except BaseException:
        if child.poll() is None:
            child.terminate()
            try: child.wait(timeout=5)
            except subprocess.TimeoutExpired:
                child.kill(); child.wait()
        raise
stream(['nvidia-smi'])
print('Shard folder:',ROOT)
print('Free local disk:',round(shutil.disk_usage(WORK).free/2**30,1),'GiB')
minimum_free_gib=5 if (WORK/'hf_cache').exists() else 35
assert shutil.disk_usage(WORK).free >= minimum_free_gib*2**30, f'Allow at least {minimum_free_gib} GiB free local disk. Incomplete caches may need further downloads.'


## 2. Create Python 3.11 and install dependencies
Downloads can take time. Logs are streamed. Do not restart the kernel after this cell; model execution uses the separate environment. Rerunning in the same VM reuses installed packages.

In [ ]:
BOOT=WORK/'uv_bootstrap'
stream([sys.executable,'-m','pip','install','--target',BOOT,'uv==0.8.22'])
UV_ENV=ENV.copy(); UV_ENV.update(PYTHONPATH=str(BOOT),UV_PYTHON_INSTALL_DIR=str(WORK/'python'),UV_PYTHON_PREFERENCE='only-managed',UV_CACHE_DIR=str(WORK/'uv_cache'))
if not PY.exists():
    stream([sys.executable,'-m','uv','python','install','3.11'],UV_ENV)
    stream([sys.executable,'-m','uv','venv','--python','3.11','--seed',VENV],UV_ENV)
stream([PY,'--version'])
stream([PY,'-m','pip','install','pip==25.0.1'])
stream([PY,'-m','pip','install','torch==2.5.1','--index-url','https://download.pytorch.org/whl/cu121'])
DEPS=['diffusers==0.35.1', 'transformers==4.49.0', 'accelerate==1.2.1', 'huggingface-hub==0.34.4', 'safetensors==0.5.3', 'tokenizers==0.21.0', 'bitsandbytes==0.45.5', 'sentencepiece==0.2.0', 'protobuf==5.29.3', 'numpy==1.26.4', 'Pillow==10.4.0', 'imageio==2.37.0', 'imageio-ffmpeg==0.6.0', 'ftfy==6.3.1', 'beautifulsoup4==4.13.4', 'scipy==1.14.1']
stream([PY,'-m','pip','install','--only-binary=:all:',*DEPS])
stream([PY,'-m','pip','check'])
stream([PY,'-c',"import torch; from diffusers import WanPipeline; assert torch.cuda.is_available(); print('GPU:',torch.cuda.get_device_name(0)); print('torch:',torch.__version__)"])


## 3. Write the embedded scripts and fixed dataset plan
No separate code upload is needed. These are the same scripts included in the download package. Changing the plan or sampling settings requires a new output dataset directory. Do not lower steps/resolution in this version.

In [ ]:
PLAN=[{'id': 'fall_00_s00', 'prompt_id': 'fall_00', 'event': 'fall', 'split': 'train', 'shard': 1, 'seed': 20261020, 'prompt': 'A realistic continuous side-view video in a bright uncluttered living room lit by a window. The scene contains a single red rubber ball, a low rectangular light-oak table, and a pale tiled floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. At the beginning the ball is already rolling toward the near edge. It rolls past the edge, visibly drops downward through the air, contacts the floor, and makes a small bounce before settling. The fall and floor contact happen within the shot. Keep the ball visible throughout this single uninterrupted action.', 'reverse_caption': 'A realistic continuous side-view video in a bright uncluttered living room lit by a window. The scene contains a single red rubber ball, a low rectangular light-oak table, and a pale tiled floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. The ball starts on the floor below the table, lifts upward from the floor through the air, reaches the tabletop edge, and rolls away from that edge onto the table. This is the backward-in-time version of a fall; upward travel and arrival on the tabletop are visible in one continuous shot.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One consistent ball visibly leaves the tabletop, travels downward and contacts the visible floor. Reject clips where the ball merely moves on the table, disappears below the frame, duplicates, or never lands.', 'context_index': 0}, {'id': 'fall_01_s00', 'prompt_id': 'fall_01', 'event': 'fall', 'split': 'train', 'shard': 2, 'seed': 20261021, 'prompt': 'A realistic continuous side-view video in a simple indoor workshop in soft daylight. The scene contains a single yellow tennis ball, a small dark wooden side table, and a gray concrete floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. At the beginning the ball is already rolling toward the near edge. It rolls past the edge, visibly drops downward through the air, contacts the floor, and makes a small bounce before settling. The fall and floor contact happen within the shot. Keep the ball visible throughout this single uninterrupted action.', 'reverse_caption': 'A realistic continuous side-view video in a simple indoor workshop in soft daylight. The scene contains a single yellow tennis ball, a small dark wooden side table, and a gray concrete floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. The ball starts on the floor below the table, lifts upward from the floor through the air, reaches the tabletop edge, and rolls away from that edge onto the table. This is the backward-in-time version of a fall; upward travel and arrival on the tabletop are visible in one continuous shot.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One consistent ball visibly leaves the tabletop, travels downward and contacts the visible floor. Reject clips where the ball merely moves on the table, disappears below the frame, duplicates, or never lands.', 'context_index': 1}, {'id': 'fall_02_s00', 'prompt_id': 'fall_02', 'event': 'fall', 'split': 'train', 'shard': 3, 'seed': 20261022, 'prompt': 'A realistic continuous side-view video in a quiet room with a plain beige wall. The scene contains a single blue rubber ball, a low white-painted wooden table, and a warm wooden floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. At the beginning the ball is already rolling toward the near edge. It rolls past the edge, visibly drops downward through the air, contacts the floor, and makes a small bounce before settling. The fall and floor contact happen within the shot. Keep the ball visible throughout this single uninterrupted action.', 'reverse_caption': 'A realistic continuous side-view video in a quiet room with a plain beige wall. The scene contains a single blue rubber ball, a low white-painted wooden table, and a warm wooden floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. The ball starts on the floor below the table, lifts upward from the floor through the air, reaches the tabletop edge, and rolls away from that edge onto the table. This is the backward-in-time version of a fall; upward travel and arrival on the tabletop are visible in one continuous shot.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One consistent ball visibly leaves the tabletop, travels downward and contacts the visible floor. Reject clips where the ball merely moves on the table, disappears below the frame, duplicates, or never lands.', 'context_index': 2}, {'id': 'fall_03_s00', 'prompt_id': 'fall_03', 'event': 'fall', 'split': 'train', 'shard': 4, 'seed': 20261023, 'prompt': 'A realistic continuous side-view video in a minimal studio with gentle side lighting. The scene contains a single orange ping-pong ball, a low walnut coffee table, and a light stone floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. At the beginning the ball is already rolling toward the near edge. It rolls past the edge, visibly drops downward through the air, contacts the floor, and makes a small bounce before settling. The fall and floor contact happen within the shot. Keep the ball visible throughout this single uninterrupted action.', 'reverse_caption': 'A realistic continuous side-view video in a minimal studio with gentle side lighting. The scene contains a single orange ping-pong ball, a low walnut coffee table, and a light stone floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. The ball starts on the floor below the table, lifts upward from the floor through the air, reaches the tabletop edge, and rolls away from that edge onto the table. This is the backward-in-time version of a fall; upward travel and arrival on the tabletop are visible in one continuous shot.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One consistent ball visibly leaves the tabletop, travels downward and contacts the visible floor. Reject clips where the ball merely moves on the table, disappears below the frame, duplicates, or never lands.', 'context_index': 3}, {'id': 'fall_04_s00', 'prompt_id': 'fall_04', 'event': 'fall', 'split': 'train', 'shard': 1, 'seed': 20261024, 'prompt': 'A realistic continuous side-view video in an empty sunlit dining area. The scene contains a single green rubber ball, a short pine table, and a cream tiled floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. At the beginning the ball is already rolling toward the near edge. It rolls past the edge, visibly drops downward through the air, contacts the floor, and makes a small bounce before settling. The fall and floor contact happen within the shot. Keep the ball visible throughout this single uninterrupted action.', 'reverse_caption': 'A realistic continuous side-view video in an empty sunlit dining area. The scene contains a single green rubber ball, a short pine table, and a cream tiled floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. The ball starts on the floor below the table, lifts upward from the floor through the air, reaches the tabletop edge, and rolls away from that edge onto the table. This is the backward-in-time version of a fall; upward travel and arrival on the tabletop are visible in one continuous shot.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One consistent ball visibly leaves the tabletop, travels downward and contacts the visible floor. Reject clips where the ball merely moves on the table, disappears below the frame, duplicates, or never lands.', 'context_index': 4}, {'id': 'fall_05_s00', 'prompt_id': 'fall_05', 'event': 'fall', 'split': 'train', 'shard': 2, 'seed': 20261025, 'prompt': 'A realistic continuous side-view video in a clean room with diffuse indoor lighting. The scene contains a single purple rubber ball, a low maple side table, and a matte gray floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. At the beginning the ball is already rolling toward the near edge. It rolls past the edge, visibly drops downward through the air, contacts the floor, and makes a small bounce before settling. The fall and floor contact happen within the shot. Keep the ball visible throughout this single uninterrupted action.', 'reverse_caption': 'A realistic continuous side-view video in a clean room with diffuse indoor lighting. The scene contains a single purple rubber ball, a low maple side table, and a matte gray floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. The ball starts on the floor below the table, lifts upward from the floor through the air, reaches the tabletop edge, and rolls away from that edge onto the table. This is the backward-in-time version of a fall; upward travel and arrival on the tabletop are visible in one continuous shot.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One consistent ball visibly leaves the tabletop, travels downward and contacts the visible floor. Reject clips where the ball merely moves on the table, disappears below the frame, duplicates, or never lands.', 'context_index': 5}, {'id': 'fall_06_s00', 'prompt_id': 'fall_06', 'event': 'fall', 'split': 'train', 'shard': 3, 'seed': 20261026, 'prompt': 'A realistic continuous side-view video in an uncluttered room with a pale blue wall. The scene contains a single white table-tennis ball, a low dark-brown wooden table, and a tan tiled floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. At the beginning the ball is already rolling toward the near edge. It rolls past the edge, visibly drops downward through the air, contacts the floor, and makes a small bounce before settling. The fall and floor contact happen within the shot. Keep the ball visible throughout this single uninterrupted action.', 'reverse_caption': 'A realistic continuous side-view video in an uncluttered room with a pale blue wall. The scene contains a single white table-tennis ball, a low dark-brown wooden table, and a tan tiled floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. The ball starts on the floor below the table, lifts upward from the floor through the air, reaches the tabletop edge, and rolls away from that edge onto the table. This is the backward-in-time version of a fall; upward travel and arrival on the tabletop are visible in one continuous shot.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One consistent ball visibly leaves the tabletop, travels downward and contacts the visible floor. Reject clips where the ball merely moves on the table, disappears below the frame, duplicates, or never lands.', 'context_index': 6}, {'id': 'fall_07_s00', 'prompt_id': 'fall_07', 'event': 'fall', 'split': 'train', 'shard': 4, 'seed': 20261027, 'prompt': 'A realistic continuous side-view video in a softly lit home interior. The scene contains a single pink rubber ball, a low square oak table, and a dark wooden floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. At the beginning the ball is already rolling toward the near edge. It rolls past the edge, visibly drops downward through the air, contacts the floor, and makes a small bounce before settling. The fall and floor contact happen within the shot. Keep the ball visible throughout this single uninterrupted action.', 'reverse_caption': 'A realistic continuous side-view video in a softly lit home interior. The scene contains a single pink rubber ball, a low square oak table, and a dark wooden floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. The ball starts on the floor below the table, lifts upward from the floor through the air, reaches the tabletop edge, and rolls away from that edge onto the table. This is the backward-in-time version of a fall; upward travel and arrival on the tabletop are visible in one continuous shot.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One consistent ball visibly leaves the tabletop, travels downward and contacts the visible floor. Reject clips where the ball merely moves on the table, disappears below the frame, duplicates, or never lands.', 'context_index': 7}, {'id': 'fall_08_s00', 'prompt_id': 'fall_08', 'event': 'fall', 'split': 'val', 'shard': 1, 'seed': 20261028, 'prompt': 'A realistic continuous side-view video in a spare room with a plain white wall. The scene contains a single turquoise rubber ball, a low birch table, and a reddish terracotta floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. At the beginning the ball is already rolling toward the near edge. It rolls past the edge, visibly drops downward through the air, contacts the floor, and makes a small bounce before settling. The fall and floor contact happen within the shot. Keep the ball visible throughout this single uninterrupted action.', 'reverse_caption': 'A realistic continuous side-view video in a spare room with a plain white wall. The scene contains a single turquoise rubber ball, a low birch table, and a reddish terracotta floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. The ball starts on the floor below the table, lifts upward from the floor through the air, reaches the tabletop edge, and rolls away from that edge onto the table. This is the backward-in-time version of a fall; upward travel and arrival on the tabletop are visible in one continuous shot.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One consistent ball visibly leaves the tabletop, travels downward and contacts the visible floor. Reject clips where the ball merely moves on the table, disappears below the frame, duplicates, or never lands.', 'context_index': 8}, {'id': 'fall_09_s00', 'prompt_id': 'fall_09', 'event': 'fall', 'split': 'test', 'shard': 2, 'seed': 20261029, 'prompt': 'A realistic continuous side-view video in a neutral room in bright indirect daylight. The scene contains a single black rubber ball, a low honey-colored wooden table, and a white tiled floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. At the beginning the ball is already rolling toward the near edge. It rolls past the edge, visibly drops downward through the air, contacts the floor, and makes a small bounce before settling. The fall and floor contact happen within the shot. Keep the ball visible throughout this single uninterrupted action.', 'reverse_caption': 'A realistic continuous side-view video in a neutral room in bright indirect daylight. The scene contains a single black rubber ball, a low honey-colored wooden table, and a white tiled floor. A fixed wide camera shows the complete near table edge, the open space below it, and the floor where the ball will land. The tabletop is low enough that its edge and landing area fit together in the frame. The ball is the only moving subject. The ball starts on the floor below the table, lifts upward from the floor through the air, reaches the tabletop edge, and rolls away from that edge onto the table. This is the backward-in-time version of a fall; upward travel and arrival on the tabletop are visible in one continuous shot.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One consistent ball visibly leaves the tabletop, travels downward and contacts the visible floor. Reject clips where the ball merely moves on the table, disappears below the frame, duplicates, or never lands.', 'context_index': 9}, {'id': 'pour_00_s00', 'prompt_id': 'pour_00', 'event': 'pour', 'split': 'train', 'shard': 1, 'seed': 20261030, 'prompt': 'A realistic close side-view video of clear water pouring from a transparent glass bottle into a transparent cylindrical drinking glass, on a light wooden counter, with soft daylight against a plain gray wall. The fixed camera includes the complete source container, its spout or mouth, the full receiving glass, and the narrow space between them. The receiving glass stands upright and begins almost empty. There are no other vessels in the scene. The source is tilted above the receiving glass. A continuous visible stream of clear water flows downward from the source mouth into the glass. The liquid level in the receiving glass visibly rises during the shot, leaving a clear difference between the beginning and end. The stream remains inside the rim. Use one continuous shot without a cut, zoom, or camera motion.', 'reverse_caption': 'A realistic close side-view video showing a transparent glass bottle and a transparent cylindrical drinking glass on a light wooden counter, with soft daylight against a plain gray wall. The camera includes both complete containers and the space between them. A visible continuous stream of clear water travels upward from the receiving glass into the tilted source container. The liquid level in the receiving glass decreases while the liquid returns to the source. This is the backward-in-time version of pouring, with the entire stream and both containers visible.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'A continuous stream enters the correct glass, both source mouth and receiving glass remain visible, and the receiving level clearly rises. Reject ambiguous level changes, a hidden source, spills, impossible streams or cuts.', 'context_index': 0}, {'id': 'pour_01_s00', 'prompt_id': 'pour_01', 'event': 'pour', 'split': 'train', 'shard': 2, 'seed': 20261031, 'prompt': 'A realistic close side-view video of orange juice pouring from a small clear glass jug into a transparent tumbler, on a white kitchen counter, with bright diffuse kitchen light. The fixed camera includes the complete source container, its spout or mouth, the full receiving glass, and the narrow space between them. The receiving glass stands upright and begins almost empty. There are no other vessels in the scene. The source is tilted above the receiving glass. A continuous visible stream of orange juice flows downward from the source mouth into the glass. The liquid level in the receiving glass visibly rises during the shot, leaving a clear difference between the beginning and end. The stream remains inside the rim. Use one continuous shot without a cut, zoom, or camera motion.', 'reverse_caption': 'A realistic close side-view video showing a small clear glass jug and a transparent tumbler on a white kitchen counter, with bright diffuse kitchen light. The camera includes both complete containers and the space between them. A visible continuous stream of orange juice travels upward from the receiving glass into the tilted source container. The liquid level in the receiving glass decreases while the liquid returns to the source. This is the backward-in-time version of pouring, with the entire stream and both containers visible.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'A continuous stream enters the correct glass, both source mouth and receiving glass remain visible, and the receiving level clearly rises. Reject ambiguous level changes, a hidden source, spills, impossible streams or cuts.', 'context_index': 1}, {'id': 'pour_02_s00', 'prompt_id': 'pour_02', 'event': 'pour', 'split': 'train', 'shard': 3, 'seed': 20261032, 'prompt': 'A realistic close side-view video of clear water pouring from a clear measuring jug into a transparent tall drinking glass, on a dark wooden counter, with gentle side lighting and a beige wall. The fixed camera includes the complete source container, its spout or mouth, the full receiving glass, and the narrow space between them. The receiving glass stands upright and begins almost empty. There are no other vessels in the scene. The source is tilted above the receiving glass. A continuous visible stream of clear water flows downward from the source mouth into the glass. The liquid level in the receiving glass visibly rises during the shot, leaving a clear difference between the beginning and end. The stream remains inside the rim. Use one continuous shot without a cut, zoom, or camera motion.', 'reverse_caption': 'A realistic close side-view video showing a clear measuring jug and a transparent tall drinking glass on a dark wooden counter, with gentle side lighting and a beige wall. The camera includes both complete containers and the space between them. A visible continuous stream of clear water travels upward from the receiving glass into the tilted source container. The liquid level in the receiving glass decreases while the liquid returns to the source. This is the backward-in-time version of pouring, with the entire stream and both containers visible.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'A continuous stream enters the correct glass, both source mouth and receiving glass remain visible, and the receiving level clearly rises. Reject ambiguous level changes, a hidden source, spills, impossible streams or cuts.', 'context_index': 2}, {'id': 'pour_03_s00', 'prompt_id': 'pour_03', 'event': 'pour', 'split': 'train', 'shard': 4, 'seed': 20261033, 'prompt': 'A realistic close side-view video of pale apple juice pouring from a transparent glass carafe into a clear straight-sided tumbler, on a light stone counter, with natural window light. The fixed camera includes the complete source container, its spout or mouth, the full receiving glass, and the narrow space between them. The receiving glass stands upright and begins almost empty. There are no other vessels in the scene. The source is tilted above the receiving glass. A continuous visible stream of pale apple juice flows downward from the source mouth into the glass. The liquid level in the receiving glass visibly rises during the shot, leaving a clear difference between the beginning and end. The stream remains inside the rim. Use one continuous shot without a cut, zoom, or camera motion.', 'reverse_caption': 'A realistic close side-view video showing a transparent glass carafe and a clear straight-sided tumbler on a light stone counter, with natural window light. The camera includes both complete containers and the space between them. A visible continuous stream of pale apple juice travels upward from the receiving glass into the tilted source container. The liquid level in the receiving glass decreases while the liquid returns to the source. This is the backward-in-time version of pouring, with the entire stream and both containers visible.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'A continuous stream enters the correct glass, both source mouth and receiving glass remain visible, and the receiving level clearly rises. Reject ambiguous level changes, a hidden source, spills, impossible streams or cuts.', 'context_index': 3}, {'id': 'pour_04_s00', 'prompt_id': 'pour_04', 'event': 'pour', 'split': 'train', 'shard': 1, 'seed': 20261034, 'prompt': 'A realistic close side-view video of clear water pouring from a small stainless-steel pitcher into a transparent cylindrical glass, on a blue countertop, with soft indoor illumination. The fixed camera includes the complete source container, its spout or mouth, the full receiving glass, and the narrow space between them. The receiving glass stands upright and begins almost empty. There are no other vessels in the scene. The source is tilted above the receiving glass. A continuous visible stream of clear water flows downward from the source mouth into the glass. The liquid level in the receiving glass visibly rises during the shot, leaving a clear difference between the beginning and end. The stream remains inside the rim. Use one continuous shot without a cut, zoom, or camera motion.', 'reverse_caption': 'A realistic close side-view video showing a small stainless-steel pitcher and a transparent cylindrical glass on a blue countertop, with soft indoor illumination. The camera includes both complete containers and the space between them. A visible continuous stream of clear water travels upward from the receiving glass into the tilted source container. The liquid level in the receiving glass decreases while the liquid returns to the source. This is the backward-in-time version of pouring, with the entire stream and both containers visible.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'A continuous stream enters the correct glass, both source mouth and receiving glass remain visible, and the receiving level clearly rises. Reject ambiguous level changes, a hidden source, spills, impossible streams or cuts.', 'context_index': 4}, {'id': 'pour_05_s00', 'prompt_id': 'pour_05', 'event': 'pour', 'split': 'train', 'shard': 2, 'seed': 20261035, 'prompt': 'A realistic close side-view video of dark tea pouring from a transparent glass teapot into a transparent glass cup, on a pale wooden counter, with warm even indoor lighting. The fixed camera includes the complete source container, its spout or mouth, the full receiving glass, and the narrow space between them. The receiving glass stands upright and begins almost empty. There are no other vessels in the scene. The source is tilted above the receiving glass. A continuous visible stream of dark tea flows downward from the source mouth into the glass. The liquid level in the receiving glass visibly rises during the shot, leaving a clear difference between the beginning and end. The stream remains inside the rim. Use one continuous shot without a cut, zoom, or camera motion.', 'reverse_caption': 'A realistic close side-view video showing a transparent glass teapot and a transparent glass cup on a pale wooden counter, with warm even indoor lighting. The camera includes both complete containers and the space between them. A visible continuous stream of dark tea travels upward from the receiving glass into the tilted source container. The liquid level in the receiving glass decreases while the liquid returns to the source. This is the backward-in-time version of pouring, with the entire stream and both containers visible.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'A continuous stream enters the correct glass, both source mouth and receiving glass remain visible, and the receiving level clearly rises. Reject ambiguous level changes, a hidden source, spills, impossible streams or cuts.', 'context_index': 5}, {'id': 'pour_06_s00', 'prompt_id': 'pour_06', 'event': 'pour', 'split': 'train', 'shard': 3, 'seed': 20261036, 'prompt': 'A realistic close side-view video of white milk pouring from a small transparent glass bottle into a transparent tumbler, on a dark stone counter, with bright neutral studio lighting. The fixed camera includes the complete source container, its spout or mouth, the full receiving glass, and the narrow space between them. The receiving glass stands upright and begins almost empty. There are no other vessels in the scene. The source is tilted above the receiving glass. A continuous visible stream of white milk flows downward from the source mouth into the glass. The liquid level in the receiving glass visibly rises during the shot, leaving a clear difference between the beginning and end. The stream remains inside the rim. Use one continuous shot without a cut, zoom, or camera motion.', 'reverse_caption': 'A realistic close side-view video showing a small transparent glass bottle and a transparent tumbler on a dark stone counter, with bright neutral studio lighting. The camera includes both complete containers and the space between them. A visible continuous stream of white milk travels upward from the receiving glass into the tilted source container. The liquid level in the receiving glass decreases while the liquid returns to the source. This is the backward-in-time version of pouring, with the entire stream and both containers visible.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'A continuous stream enters the correct glass, both source mouth and receiving glass remain visible, and the receiving level clearly rises. Reject ambiguous level changes, a hidden source, spills, impossible streams or cuts.', 'context_index': 6}, {'id': 'pour_07_s00', 'prompt_id': 'pour_07', 'event': 'pour', 'split': 'train', 'shard': 4, 'seed': 20261037, 'prompt': 'A realistic close side-view video of red fruit juice pouring from a small clear glass pitcher into a clear cylindrical glass, on a cream countertop, with diffuse daylight. The fixed camera includes the complete source container, its spout or mouth, the full receiving glass, and the narrow space between them. The receiving glass stands upright and begins almost empty. There are no other vessels in the scene. The source is tilted above the receiving glass. A continuous visible stream of red fruit juice flows downward from the source mouth into the glass. The liquid level in the receiving glass visibly rises during the shot, leaving a clear difference between the beginning and end. The stream remains inside the rim. Use one continuous shot without a cut, zoom, or camera motion.', 'reverse_caption': 'A realistic close side-view video showing a small clear glass pitcher and a clear cylindrical glass on a cream countertop, with diffuse daylight. The camera includes both complete containers and the space between them. A visible continuous stream of red fruit juice travels upward from the receiving glass into the tilted source container. The liquid level in the receiving glass decreases while the liquid returns to the source. This is the backward-in-time version of pouring, with the entire stream and both containers visible.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'A continuous stream enters the correct glass, both source mouth and receiving glass remain visible, and the receiving level clearly rises. Reject ambiguous level changes, a hidden source, spills, impossible streams or cuts.', 'context_index': 7}, {'id': 'pour_08_s00', 'prompt_id': 'pour_08', 'event': 'pour', 'split': 'val', 'shard': 2, 'seed': 20261038, 'prompt': 'A realistic close side-view video of clear water pouring from a transparent square glass bottle into a clear wide tumbler, on a terracotta-colored counter, with gentle morning window light. The fixed camera includes the complete source container, its spout or mouth, the full receiving glass, and the narrow space between them. The receiving glass stands upright and begins almost empty. There are no other vessels in the scene. The source is tilted above the receiving glass. A continuous visible stream of clear water flows downward from the source mouth into the glass. The liquid level in the receiving glass visibly rises during the shot, leaving a clear difference between the beginning and end. The stream remains inside the rim. Use one continuous shot without a cut, zoom, or camera motion.', 'reverse_caption': 'A realistic close side-view video showing a transparent square glass bottle and a clear wide tumbler on a terracotta-colored counter, with gentle morning window light. The camera includes both complete containers and the space between them. A visible continuous stream of clear water travels upward from the receiving glass into the tilted source container. The liquid level in the receiving glass decreases while the liquid returns to the source. This is the backward-in-time version of pouring, with the entire stream and both containers visible.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'A continuous stream enters the correct glass, both source mouth and receiving glass remain visible, and the receiving level clearly rises. Reject ambiguous level changes, a hidden source, spills, impossible streams or cuts.', 'context_index': 8}, {'id': 'pour_09_s00', 'prompt_id': 'pour_09', 'event': 'pour', 'split': 'test', 'shard': 3, 'seed': 20261039, 'prompt': 'A realistic close side-view video of pale lemonade pouring from a clear ribbed glass jug into a transparent straight-sided glass, on a green countertop, with soft neutral lighting. The fixed camera includes the complete source container, its spout or mouth, the full receiving glass, and the narrow space between them. The receiving glass stands upright and begins almost empty. There are no other vessels in the scene. The source is tilted above the receiving glass. A continuous visible stream of pale lemonade flows downward from the source mouth into the glass. The liquid level in the receiving glass visibly rises during the shot, leaving a clear difference between the beginning and end. The stream remains inside the rim. Use one continuous shot without a cut, zoom, or camera motion.', 'reverse_caption': 'A realistic close side-view video showing a clear ribbed glass jug and a transparent straight-sided glass on a green countertop, with soft neutral lighting. The camera includes both complete containers and the space between them. A visible continuous stream of pale lemonade travels upward from the receiving glass into the tilted source container. The liquid level in the receiving glass decreases while the liquid returns to the source. This is the backward-in-time version of pouring, with the entire stream and both containers visible.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'A continuous stream enters the correct glass, both source mouth and receiving glass remain visible, and the receiving level clearly rises. Reject ambiguous level changes, a hidden source, spills, impossible streams or cuts.', 'context_index': 9}, {'id': 'ink_00_s00', 'prompt_id': 'ink_00', 'event': 'ink', 'split': 'train', 'shard': 1, 'seed': 20261040, 'prompt': 'A realistic close side-view video of blue ink dispersing in still clear water inside a clear square glass tank, standing on a pale wooden surface, with a plain white background and soft evenly distributed daylight. A fixed camera shows the full water-filled vessel, its water surface, and the clear water below. The water begins transparent. A single small dropper tip appears just above the water surface near the upper center; the vessel stays stationary. One concentrated drop of blue ink enters the water near the beginning of the shot. The drop sinks into the water and expands into visible branching wisps and a larger soft cloud. The colored region becomes visibly wider and more spread out over time while clear water remains around it. Keep the initial compact drop and the later dispersed cloud distinguishable. No stirring, pouring, camera motion or scene cuts.', 'reverse_caption': 'A realistic close side-view video showing a clear square glass tank filled with water and visible blue ink, standing on a pale wooden surface, with a plain white background and soft evenly distributed daylight. The entire vessel and water surface remain visible. The dispersed wisps of blue ink contract through the clear water and gather into a small concentrated drop near the water surface. The colored region becomes narrower and more compact over time. This is the backward-in-time version of ink dispersion; the vessel and camera remain stationary.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'The colored region visibly expands from a compact drop into branching diffuse wisps in clear water. Reject an already uniform colored vessel, a motionless cloud, stirring, or cuts.', 'context_index': 0}, {'id': 'ink_01_s00', 'prompt_id': 'ink_01', 'event': 'ink', 'split': 'train', 'shard': 2, 'seed': 20261041, 'prompt': 'A realistic close side-view video of red ink dispersing in still clear water inside a clear cylindrical glass vessel, standing on a white surface, with a light gray background and diffuse studio lighting. A fixed camera shows the full water-filled vessel, its water surface, and the clear water below. The water begins transparent. A single small dropper tip appears just above the water surface near the upper center; the vessel stays stationary. One concentrated drop of red ink enters the water near the beginning of the shot. The drop sinks into the water and expands into visible branching wisps and a larger soft cloud. The colored region becomes visibly wider and more spread out over time while clear water remains around it. Keep the initial compact drop and the later dispersed cloud distinguishable. No stirring, pouring, camera motion or scene cuts.', 'reverse_caption': 'A realistic close side-view video showing a clear cylindrical glass vessel filled with water and visible red ink, standing on a white surface, with a light gray background and diffuse studio lighting. The entire vessel and water surface remain visible. The dispersed wisps of red ink contract through the clear water and gather into a small concentrated drop near the water surface. The colored region becomes narrower and more compact over time. This is the backward-in-time version of ink dispersion; the vessel and camera remain stationary.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'The colored region visibly expands from a compact drop into branching diffuse wisps in clear water. Reject an already uniform colored vessel, a motionless cloud, stirring, or cuts.', 'context_index': 1}, {'id': 'ink_02_s00', 'prompt_id': 'ink_02', 'event': 'ink', 'split': 'train', 'shard': 3, 'seed': 20261042, 'prompt': 'A realistic close side-view video of green ink dispersing in still clear water inside a transparent rectangular water tank, standing on a dark wooden surface, with a cream background and gentle side lighting. A fixed camera shows the full water-filled vessel, its water surface, and the clear water below. The water begins transparent. A single small dropper tip appears just above the water surface near the upper center; the vessel stays stationary. One concentrated drop of green ink enters the water near the beginning of the shot. The drop sinks into the water and expands into visible branching wisps and a larger soft cloud. The colored region becomes visibly wider and more spread out over time while clear water remains around it. Keep the initial compact drop and the later dispersed cloud distinguishable. No stirring, pouring, camera motion or scene cuts.', 'reverse_caption': 'A realistic close side-view video showing a transparent rectangular water tank filled with water and visible green ink, standing on a dark wooden surface, with a cream background and gentle side lighting. The entire vessel and water surface remain visible. The dispersed wisps of green ink contract through the clear water and gather into a small concentrated drop near the water surface. The colored region becomes narrower and more compact over time. This is the backward-in-time version of ink dispersion; the vessel and camera remain stationary.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'The colored region visibly expands from a compact drop into branching diffuse wisps in clear water. Reject an already uniform colored vessel, a motionless cloud, stirring, or cuts.', 'context_index': 2}, {'id': 'ink_03_s00', 'prompt_id': 'ink_03', 'event': 'ink', 'split': 'train', 'shard': 4, 'seed': 20261043, 'prompt': 'A realistic close side-view video of purple ink dispersing in still clear water inside a clear tall glass beaker, standing on a light stone surface, with a pale blue background and neutral indoor light. A fixed camera shows the full water-filled vessel, its water surface, and the clear water below. The water begins transparent. A single small dropper tip appears just above the water surface near the upper center; the vessel stays stationary. One concentrated drop of purple ink enters the water near the beginning of the shot. The drop sinks into the water and expands into visible branching wisps and a larger soft cloud. The colored region becomes visibly wider and more spread out over time while clear water remains around it. Keep the initial compact drop and the later dispersed cloud distinguishable. No stirring, pouring, camera motion or scene cuts.', 'reverse_caption': 'A realistic close side-view video showing a clear tall glass beaker filled with water and visible purple ink, standing on a light stone surface, with a pale blue background and neutral indoor light. The entire vessel and water surface remain visible. The dispersed wisps of purple ink contract through the clear water and gather into a small concentrated drop near the water surface. The colored region becomes narrower and more compact over time. This is the backward-in-time version of ink dispersion; the vessel and camera remain stationary.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'The colored region visibly expands from a compact drop into branching diffuse wisps in clear water. Reject an already uniform colored vessel, a motionless cloud, stirring, or cuts.', 'context_index': 3}, {'id': 'ink_04_s00', 'prompt_id': 'ink_04', 'event': 'ink', 'split': 'train', 'shard': 1, 'seed': 20261044, 'prompt': 'A realistic close side-view video of black ink dispersing in still clear water inside a clear cube-shaped glass tank, standing on a gray surface, with a white background and bright soft light. A fixed camera shows the full water-filled vessel, its water surface, and the clear water below. The water begins transparent. A single small dropper tip appears just above the water surface near the upper center; the vessel stays stationary. One concentrated drop of black ink enters the water near the beginning of the shot. The drop sinks into the water and expands into visible branching wisps and a larger soft cloud. The colored region becomes visibly wider and more spread out over time while clear water remains around it. Keep the initial compact drop and the later dispersed cloud distinguishable. No stirring, pouring, camera motion or scene cuts.', 'reverse_caption': 'A realistic close side-view video showing a clear cube-shaped glass tank filled with water and visible black ink, standing on a gray surface, with a white background and bright soft light. The entire vessel and water surface remain visible. The dispersed wisps of black ink contract through the clear water and gather into a small concentrated drop near the water surface. The colored region becomes narrower and more compact over time. This is the backward-in-time version of ink dispersion; the vessel and camera remain stationary.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'The colored region visibly expands from a compact drop into branching diffuse wisps in clear water. Reject an already uniform colored vessel, a motionless cloud, stirring, or cuts.', 'context_index': 4}, {'id': 'ink_05_s00', 'prompt_id': 'ink_05', 'event': 'ink', 'split': 'train', 'shard': 2, 'seed': 20261045, 'prompt': 'A realistic close side-view video of orange ink dispersing in still clear water inside a transparent cylindrical vessel, standing on a beige surface, with a pale gray background and even window light. A fixed camera shows the full water-filled vessel, its water surface, and the clear water below. The water begins transparent. A single small dropper tip appears just above the water surface near the upper center; the vessel stays stationary. One concentrated drop of orange ink enters the water near the beginning of the shot. The drop sinks into the water and expands into visible branching wisps and a larger soft cloud. The colored region becomes visibly wider and more spread out over time while clear water remains around it. Keep the initial compact drop and the later dispersed cloud distinguishable. No stirring, pouring, camera motion or scene cuts.', 'reverse_caption': 'A realistic close side-view video showing a transparent cylindrical vessel filled with water and visible orange ink, standing on a beige surface, with a pale gray background and even window light. The entire vessel and water surface remain visible. The dispersed wisps of orange ink contract through the clear water and gather into a small concentrated drop near the water surface. The colored region becomes narrower and more compact over time. This is the backward-in-time version of ink dispersion; the vessel and camera remain stationary.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'The colored region visibly expands from a compact drop into branching diffuse wisps in clear water. Reject an already uniform colored vessel, a motionless cloud, stirring, or cuts.', 'context_index': 5}, {'id': 'ink_06_s00', 'prompt_id': 'ink_06', 'event': 'ink', 'split': 'train', 'shard': 3, 'seed': 20261046, 'prompt': 'A realistic close side-view video of cyan ink dispersing in still clear water inside a clear rectangular glass tank, standing on a brown wooden surface, with a warm white background and diffuse daylight. A fixed camera shows the full water-filled vessel, its water surface, and the clear water below. The water begins transparent. A single small dropper tip appears just above the water surface near the upper center; the vessel stays stationary. One concentrated drop of cyan ink enters the water near the beginning of the shot. The drop sinks into the water and expands into visible branching wisps and a larger soft cloud. The colored region becomes visibly wider and more spread out over time while clear water remains around it. Keep the initial compact drop and the later dispersed cloud distinguishable. No stirring, pouring, camera motion or scene cuts.', 'reverse_caption': 'A realistic close side-view video showing a clear rectangular glass tank filled with water and visible cyan ink, standing on a brown wooden surface, with a warm white background and diffuse daylight. The entire vessel and water surface remain visible. The dispersed wisps of cyan ink contract through the clear water and gather into a small concentrated drop near the water surface. The colored region becomes narrower and more compact over time. This is the backward-in-time version of ink dispersion; the vessel and camera remain stationary.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'The colored region visibly expands from a compact drop into branching diffuse wisps in clear water. Reject an already uniform colored vessel, a motionless cloud, stirring, or cuts.', 'context_index': 6}, {'id': 'ink_07_s00', 'prompt_id': 'ink_07', 'event': 'ink', 'split': 'train', 'shard': 4, 'seed': 20261047, 'prompt': 'A realistic close side-view video of magenta ink dispersing in still clear water inside a clear broad glass beaker, standing on a white countertop, with a pale cream background and soft studio lighting. A fixed camera shows the full water-filled vessel, its water surface, and the clear water below. The water begins transparent. A single small dropper tip appears just above the water surface near the upper center; the vessel stays stationary. One concentrated drop of magenta ink enters the water near the beginning of the shot. The drop sinks into the water and expands into visible branching wisps and a larger soft cloud. The colored region becomes visibly wider and more spread out over time while clear water remains around it. Keep the initial compact drop and the later dispersed cloud distinguishable. No stirring, pouring, camera motion or scene cuts.', 'reverse_caption': 'A realistic close side-view video showing a clear broad glass beaker filled with water and visible magenta ink, standing on a white countertop, with a pale cream background and soft studio lighting. The entire vessel and water surface remain visible. The dispersed wisps of magenta ink contract through the clear water and gather into a small concentrated drop near the water surface. The colored region becomes narrower and more compact over time. This is the backward-in-time version of ink dispersion; the vessel and camera remain stationary.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'The colored region visibly expands from a compact drop into branching diffuse wisps in clear water. Reject an already uniform colored vessel, a motionless cloud, stirring, or cuts.', 'context_index': 7}, {'id': 'ink_08_s00', 'prompt_id': 'ink_08', 'event': 'ink', 'split': 'val', 'shard': 3, 'seed': 20261048, 'prompt': 'A realistic close side-view video of dark violet ink dispersing in still clear water inside a transparent square glass vessel, standing on a tan stone surface, with a very light green background and gentle neutral light. A fixed camera shows the full water-filled vessel, its water surface, and the clear water below. The water begins transparent. A single small dropper tip appears just above the water surface near the upper center; the vessel stays stationary. One concentrated drop of dark violet ink enters the water near the beginning of the shot. The drop sinks into the water and expands into visible branching wisps and a larger soft cloud. The colored region becomes visibly wider and more spread out over time while clear water remains around it. Keep the initial compact drop and the later dispersed cloud distinguishable. No stirring, pouring, camera motion or scene cuts.', 'reverse_caption': 'A realistic close side-view video showing a transparent square glass vessel filled with water and visible dark violet ink, standing on a tan stone surface, with a very light green background and gentle neutral light. The entire vessel and water surface remain visible. The dispersed wisps of dark violet ink contract through the clear water and gather into a small concentrated drop near the water surface. The colored region becomes narrower and more compact over time. This is the backward-in-time version of ink dispersion; the vessel and camera remain stationary.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'The colored region visibly expands from a compact drop into branching diffuse wisps in clear water. Reject an already uniform colored vessel, a motionless cloud, stirring, or cuts.', 'context_index': 8}, {'id': 'ink_09_s00', 'prompt_id': 'ink_09', 'event': 'ink', 'split': 'test', 'shard': 4, 'seed': 20261049, 'prompt': 'A realistic close side-view video of deep blue ink dispersing in still clear water inside a clear tall rectangular tank, standing on a dark gray surface, with a pale peach background and bright indirect daylight. A fixed camera shows the full water-filled vessel, its water surface, and the clear water below. The water begins transparent. A single small dropper tip appears just above the water surface near the upper center; the vessel stays stationary. One concentrated drop of deep blue ink enters the water near the beginning of the shot. The drop sinks into the water and expands into visible branching wisps and a larger soft cloud. The colored region becomes visibly wider and more spread out over time while clear water remains around it. Keep the initial compact drop and the later dispersed cloud distinguishable. No stirring, pouring, camera motion or scene cuts.', 'reverse_caption': 'A realistic close side-view video showing a clear tall rectangular tank filled with water and visible deep blue ink, standing on a dark gray surface, with a pale peach background and bright indirect daylight. The entire vessel and water surface remain visible. The dispersed wisps of deep blue ink contract through the clear water and gather into a small concentrated drop near the water surface. The colored region becomes narrower and more compact over time. This is the backward-in-time version of ink dispersion; the vessel and camera remain stationary.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'The colored region visibly expands from a compact drop into branching diffuse wisps in clear water. Reject an already uniform colored vessel, a motionless cloud, stirring, or cuts.', 'context_index': 9}, {'id': 'inflate_00_s00', 'prompt_id': 'inflate_00', 'event': 'inflate', 'split': 'train', 'shard': 1, 'seed': 20261050, 'prompt': 'A realistic close side-view video of one red latex balloon being inflated above a matte white tabletop, with a plain gray background and soft indoor light. The balloon neck is attached to a short stationary air nozzle near the center of the tabletop. A small wrinkled balloon is visible at the beginning. The fixed wide framing leaves ample empty space around the balloon for its final expanded shape, with the entire balloon and nozzle always in view. Air enters through the nozzle and the single red balloon visibly expands during the shot. Its body grows from small and wrinkled into a much larger smooth rounded balloon, while its neck stays anchored at the same place. The increase in size happens continuously within this one shot. Preserve the same balloon color and identity. No camera movement, zoom, jump cuts, bursting, extra balloons or hands covering the balloon.', 'reverse_caption': 'A realistic close side-view video showing one red latex balloon attached to a stationary air nozzle above a matte white tabletop, with a plain gray background and soft indoor light. The fixed framing leaves the entire balloon and nozzle visible. The large smooth red balloon steadily shrinks and becomes small and wrinkled as air leaves through the same fixed nozzle. Its neck stays anchored, and the balloon keeps the same color and identity. This is the backward-in-time version of balloon inflation, with the full balloon visible throughout.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One anchored balloon smoothly and substantially increases in size with a stationary camera and fixed neck. Reject a mere camera zoom, floating balloon, changing identity, duplication, bursting or static size.', 'context_index': 0}, {'id': 'inflate_01_s00', 'prompt_id': 'inflate_01', 'event': 'inflate', 'split': 'train', 'shard': 2, 'seed': 20261051, 'prompt': 'A realistic close side-view video of one blue latex balloon being inflated above a pale wooden tabletop, with a cream background and natural window light. The balloon neck is attached to a short stationary air nozzle near the center of the tabletop. A small wrinkled balloon is visible at the beginning. The fixed wide framing leaves ample empty space around the balloon for its final expanded shape, with the entire balloon and nozzle always in view. Air enters through the nozzle and the single blue balloon visibly expands during the shot. Its body grows from small and wrinkled into a much larger smooth rounded balloon, while its neck stays anchored at the same place. The increase in size happens continuously within this one shot. Preserve the same balloon color and identity. No camera movement, zoom, jump cuts, bursting, extra balloons or hands covering the balloon.', 'reverse_caption': 'A realistic close side-view video showing one blue latex balloon attached to a stationary air nozzle above a pale wooden tabletop, with a cream background and natural window light. The fixed framing leaves the entire balloon and nozzle visible. The large smooth blue balloon steadily shrinks and becomes small and wrinkled as air leaves through the same fixed nozzle. Its neck stays anchored, and the balloon keeps the same color and identity. This is the backward-in-time version of balloon inflation, with the full balloon visible throughout.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One anchored balloon smoothly and substantially increases in size with a stationary camera and fixed neck. Reject a mere camera zoom, floating balloon, changing identity, duplication, bursting or static size.', 'context_index': 1}, {'id': 'inflate_02_s00', 'prompt_id': 'inflate_02', 'event': 'inflate', 'split': 'train', 'shard': 3, 'seed': 20261052, 'prompt': 'A realistic close side-view video of one yellow latex balloon being inflated above a dark wooden tabletop, with a pale blue background and diffuse studio light. The balloon neck is attached to a short stationary air nozzle near the center of the tabletop. A small wrinkled balloon is visible at the beginning. The fixed wide framing leaves ample empty space around the balloon for its final expanded shape, with the entire balloon and nozzle always in view. Air enters through the nozzle and the single yellow balloon visibly expands during the shot. Its body grows from small and wrinkled into a much larger smooth rounded balloon, while its neck stays anchored at the same place. The increase in size happens continuously within this one shot. Preserve the same balloon color and identity. No camera movement, zoom, jump cuts, bursting, extra balloons or hands covering the balloon.', 'reverse_caption': 'A realistic close side-view video showing one yellow latex balloon attached to a stationary air nozzle above a dark wooden tabletop, with a pale blue background and diffuse studio light. The fixed framing leaves the entire balloon and nozzle visible. The large smooth yellow balloon steadily shrinks and becomes small and wrinkled as air leaves through the same fixed nozzle. Its neck stays anchored, and the balloon keeps the same color and identity. This is the backward-in-time version of balloon inflation, with the full balloon visible throughout.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One anchored balloon smoothly and substantially increases in size with a stationary camera and fixed neck. Reject a mere camera zoom, floating balloon, changing identity, duplication, bursting or static size.', 'context_index': 2}, {'id': 'inflate_03_s00', 'prompt_id': 'inflate_03', 'event': 'inflate', 'split': 'train', 'shard': 4, 'seed': 20261053, 'prompt': 'A realistic close side-view video of one green latex balloon being inflated above a white stone tabletop, with a beige background and even daylight. The balloon neck is attached to a short stationary air nozzle near the center of the tabletop. A small wrinkled balloon is visible at the beginning. The fixed wide framing leaves ample empty space around the balloon for its final expanded shape, with the entire balloon and nozzle always in view. Air enters through the nozzle and the single green balloon visibly expands during the shot. Its body grows from small and wrinkled into a much larger smooth rounded balloon, while its neck stays anchored at the same place. The increase in size happens continuously within this one shot. Preserve the same balloon color and identity. No camera movement, zoom, jump cuts, bursting, extra balloons or hands covering the balloon.', 'reverse_caption': 'A realistic close side-view video showing one green latex balloon attached to a stationary air nozzle above a white stone tabletop, with a beige background and even daylight. The fixed framing leaves the entire balloon and nozzle visible. The large smooth green balloon steadily shrinks and becomes small and wrinkled as air leaves through the same fixed nozzle. Its neck stays anchored, and the balloon keeps the same color and identity. This is the backward-in-time version of balloon inflation, with the full balloon visible throughout.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One anchored balloon smoothly and substantially increases in size with a stationary camera and fixed neck. Reject a mere camera zoom, floating balloon, changing identity, duplication, bursting or static size.', 'context_index': 3}, {'id': 'inflate_04_s00', 'prompt_id': 'inflate_04', 'event': 'inflate', 'split': 'train', 'shard': 1, 'seed': 20261054, 'prompt': 'A realistic close side-view video of one purple latex balloon being inflated above a gray tabletop, with a white background and soft side lighting. The balloon neck is attached to a short stationary air nozzle near the center of the tabletop. A small wrinkled balloon is visible at the beginning. The fixed wide framing leaves ample empty space around the balloon for its final expanded shape, with the entire balloon and nozzle always in view. Air enters through the nozzle and the single purple balloon visibly expands during the shot. Its body grows from small and wrinkled into a much larger smooth rounded balloon, while its neck stays anchored at the same place. The increase in size happens continuously within this one shot. Preserve the same balloon color and identity. No camera movement, zoom, jump cuts, bursting, extra balloons or hands covering the balloon.', 'reverse_caption': 'A realistic close side-view video showing one purple latex balloon attached to a stationary air nozzle above a gray tabletop, with a white background and soft side lighting. The fixed framing leaves the entire balloon and nozzle visible. The large smooth purple balloon steadily shrinks and becomes small and wrinkled as air leaves through the same fixed nozzle. Its neck stays anchored, and the balloon keeps the same color and identity. This is the backward-in-time version of balloon inflation, with the full balloon visible throughout.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One anchored balloon smoothly and substantially increases in size with a stationary camera and fixed neck. Reject a mere camera zoom, floating balloon, changing identity, duplication, bursting or static size.', 'context_index': 4}, {'id': 'inflate_05_s00', 'prompt_id': 'inflate_05', 'event': 'inflate', 'split': 'train', 'shard': 2, 'seed': 20261055, 'prompt': 'A realistic close side-view video of one orange latex balloon being inflated above a light oak tabletop, with a light gray background and neutral studio light. The balloon neck is attached to a short stationary air nozzle near the center of the tabletop. A small wrinkled balloon is visible at the beginning. The fixed wide framing leaves ample empty space around the balloon for its final expanded shape, with the entire balloon and nozzle always in view. Air enters through the nozzle and the single orange balloon visibly expands during the shot. Its body grows from small and wrinkled into a much larger smooth rounded balloon, while its neck stays anchored at the same place. The increase in size happens continuously within this one shot. Preserve the same balloon color and identity. No camera movement, zoom, jump cuts, bursting, extra balloons or hands covering the balloon.', 'reverse_caption': 'A realistic close side-view video showing one orange latex balloon attached to a stationary air nozzle above a light oak tabletop, with a light gray background and neutral studio light. The fixed framing leaves the entire balloon and nozzle visible. The large smooth orange balloon steadily shrinks and becomes small and wrinkled as air leaves through the same fixed nozzle. Its neck stays anchored, and the balloon keeps the same color and identity. This is the backward-in-time version of balloon inflation, with the full balloon visible throughout.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One anchored balloon smoothly and substantially increases in size with a stationary camera and fixed neck. Reject a mere camera zoom, floating balloon, changing identity, duplication, bursting or static size.', 'context_index': 5}, {'id': 'inflate_06_s00', 'prompt_id': 'inflate_06', 'event': 'inflate', 'split': 'train', 'shard': 3, 'seed': 20261056, 'prompt': 'A realistic close side-view video of one pink latex balloon being inflated above a dark stone tabletop, with a pale cream background and bright indirect daylight. The balloon neck is attached to a short stationary air nozzle near the center of the tabletop. A small wrinkled balloon is visible at the beginning. The fixed wide framing leaves ample empty space around the balloon for its final expanded shape, with the entire balloon and nozzle always in view. Air enters through the nozzle and the single pink balloon visibly expands during the shot. Its body grows from small and wrinkled into a much larger smooth rounded balloon, while its neck stays anchored at the same place. The increase in size happens continuously within this one shot. Preserve the same balloon color and identity. No camera movement, zoom, jump cuts, bursting, extra balloons or hands covering the balloon.', 'reverse_caption': 'A realistic close side-view video showing one pink latex balloon attached to a stationary air nozzle above a dark stone tabletop, with a pale cream background and bright indirect daylight. The fixed framing leaves the entire balloon and nozzle visible. The large smooth pink balloon steadily shrinks and becomes small and wrinkled as air leaves through the same fixed nozzle. Its neck stays anchored, and the balloon keeps the same color and identity. This is the backward-in-time version of balloon inflation, with the full balloon visible throughout.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One anchored balloon smoothly and substantially increases in size with a stationary camera and fixed neck. Reject a mere camera zoom, floating balloon, changing identity, duplication, bursting or static size.', 'context_index': 6}, {'id': 'inflate_07_s00', 'prompt_id': 'inflate_07', 'event': 'inflate', 'split': 'train', 'shard': 4, 'seed': 20261057, 'prompt': 'A realistic close side-view video of one white latex balloon being inflated above a blue tabletop, with a dark gray background and gentle indoor light. The balloon neck is attached to a short stationary air nozzle near the center of the tabletop. A small wrinkled balloon is visible at the beginning. The fixed wide framing leaves ample empty space around the balloon for its final expanded shape, with the entire balloon and nozzle always in view. Air enters through the nozzle and the single white balloon visibly expands during the shot. Its body grows from small and wrinkled into a much larger smooth rounded balloon, while its neck stays anchored at the same place. The increase in size happens continuously within this one shot. Preserve the same balloon color and identity. No camera movement, zoom, jump cuts, bursting, extra balloons or hands covering the balloon.', 'reverse_caption': 'A realistic close side-view video showing one white latex balloon attached to a stationary air nozzle above a blue tabletop, with a dark gray background and gentle indoor light. The fixed framing leaves the entire balloon and nozzle visible. The large smooth white balloon steadily shrinks and becomes small and wrinkled as air leaves through the same fixed nozzle. Its neck stays anchored, and the balloon keeps the same color and identity. This is the backward-in-time version of balloon inflation, with the full balloon visible throughout.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One anchored balloon smoothly and substantially increases in size with a stationary camera and fixed neck. Reject a mere camera zoom, floating balloon, changing identity, duplication, bursting or static size.', 'context_index': 7}, {'id': 'inflate_08_s00', 'prompt_id': 'inflate_08', 'event': 'inflate', 'split': 'val', 'shard': 4, 'seed': 20261058, 'prompt': 'A realistic close side-view video of one turquoise latex balloon being inflated above a terracotta-colored tabletop, with a pale white background and diffuse morning light. The balloon neck is attached to a short stationary air nozzle near the center of the tabletop. A small wrinkled balloon is visible at the beginning. The fixed wide framing leaves ample empty space around the balloon for its final expanded shape, with the entire balloon and nozzle always in view. Air enters through the nozzle and the single turquoise balloon visibly expands during the shot. Its body grows from small and wrinkled into a much larger smooth rounded balloon, while its neck stays anchored at the same place. The increase in size happens continuously within this one shot. Preserve the same balloon color and identity. No camera movement, zoom, jump cuts, bursting, extra balloons or hands covering the balloon.', 'reverse_caption': 'A realistic close side-view video showing one turquoise latex balloon attached to a stationary air nozzle above a terracotta-colored tabletop, with a pale white background and diffuse morning light. The fixed framing leaves the entire balloon and nozzle visible. The large smooth turquoise balloon steadily shrinks and becomes small and wrinkled as air leaves through the same fixed nozzle. Its neck stays anchored, and the balloon keeps the same color and identity. This is the backward-in-time version of balloon inflation, with the full balloon visible throughout.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One anchored balloon smoothly and substantially increases in size with a stationary camera and fixed neck. Reject a mere camera zoom, floating balloon, changing identity, duplication, bursting or static size.', 'context_index': 8}, {'id': 'inflate_09_s00', 'prompt_id': 'inflate_09', 'event': 'inflate', 'split': 'test', 'shard': 1, 'seed': 20261059, 'prompt': 'A realistic close side-view video of one black latex balloon being inflated above a pale birch tabletop, with a light yellow background and even soft illumination. The balloon neck is attached to a short stationary air nozzle near the center of the tabletop. A small wrinkled balloon is visible at the beginning. The fixed wide framing leaves ample empty space around the balloon for its final expanded shape, with the entire balloon and nozzle always in view. Air enters through the nozzle and the single black balloon visibly expands during the shot. Its body grows from small and wrinkled into a much larger smooth rounded balloon, while its neck stays anchored at the same place. The increase in size happens continuously within this one shot. Preserve the same balloon color and identity. No camera movement, zoom, jump cuts, bursting, extra balloons or hands covering the balloon.', 'reverse_caption': 'A realistic close side-view video showing one black latex balloon attached to a stationary air nozzle above a pale birch tabletop, with a light yellow background and even soft illumination. The fixed framing leaves the entire balloon and nozzle visible. The large smooth black balloon steadily shrinks and becomes small and wrinkled as air leaves through the same fixed nozzle. Its neck stays anchored, and the balloon keeps the same color and identity. This is the backward-in-time version of balloon inflation, with the full balloon visible throughout.', 'negative_prompt': 'frozen action, still photograph, jump cuts, camera shake, zoom, duplicated main subject, extra balls, extra balloons, deformed objects, impossible object disappearance, flickering, subtitles, text, logos, watermark, blurry details', 'acceptance': 'One anchored balloon smoothly and substantially increases in size with a stationary camera and fixed neck. Reject a mere camera zoom, floating balloon, changing identity, duplication, bursting or static size.', 'context_index': 9}]
(WORK/'prompts_40.json').write_text(json.dumps(PLAN,indent=2))
(WORK/'wan_shards_common.py').write_text('"""Pure Python inventory, checksums, packaging and review utilities."""\nimport csv\nimport hashlib\nimport html\nimport json\nimport os\nimport zipfile\nfrom pathlib import Path\n\nSCHEMA = \'wan-reverse-pilot-v1\'\nMODEL = \'Wan-AI/Wan2.1-T2V-1.3B-Diffusers\'\nREVISION = \'0fad780a534b6463e45facd96134c9f345acfa5b\'\nGENERATION = dict(height=480, width=832, frames=81, fps=16, steps=50,\n                  guidance=6.0, flow_shift=8.0, transformer_precision=\'fp16\',\n                  vae_precision=\'fp32\', text_encoder=\'nf4-fp32-512\')\n\ndef canonical(value):\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\n\ndef digest(value):\n    return hashlib.sha256(canonical(value).encode()).hexdigest()\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp\')\n    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n    os.replace(tmp, path)\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\ndef shard_plan(plan, shard):\n    events = [\'fall\', \'pour\', \'ink\', \'inflate\']\n    return sorted([r for r in plan if r[\'shard\'] == shard], key=lambda r: ((events.index(r[\'event\']) - (shard - 1)) % 4, r[\'context_index\']))\n\ndef initialize(root, shard, plan):\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    rows = shard_plan(plan, shard)\n    assert len(rows) == 10 and len({r[\'id\'] for r in rows}) == 10\n    config = dict(schema=SCHEMA, model=MODEL, revision=REVISION,\n                  generation=GENERATION, shard=shard, prompts=rows)\n    config[\'fingerprint\'] = digest(config)\n    existing = root / \'dataset_config.json\'\n    if existing.exists() and read_json(existing) != config:\n        raise ValueError(\'Existing shard configuration differs. Use a new output root; do not mix runs.\')\n    atomic_json(existing, config)\n    return config\n\ndef verify_record(root, record, config=None, check_hash=True):\n    root = Path(root).resolve()\n    if config:\n        expected = {r[\'id\']: r for r in config[\'prompts\']}\n        base = expected.get(record.get(\'id\'))\n        if not base:\n            raise ValueError(\'Unknown candidate ID\')\n        for k, v in base.items():\n            if record.get(k) != v:\n                raise ValueError(f\'Candidate label mismatch: {k}\')\n        if record.get(\'dataset_fingerprint\') != config[\'fingerprint\']:\n            raise ValueError(\'Dataset fingerprint mismatch\')\n        if record.get(\'model\') != config[\'model\'] or record.get(\'revision\') != config[\'revision\']:\n            raise ValueError(\'Teacher mismatch\')\n        if record.get(\'generation\') != config[\'generation\']:\n            raise ValueError(\'Sampling configuration mismatch\')\n    required = {\'forward\', \'reverse\', \'frames_uint8\', \'forward_latent\', \'embedding\', \'negative_embedding\'}\n    if not required.issubset(record.get(\'assets\', {})):\n        raise ValueError(\'Required assets missing from record\')\n    for asset in record[\'assets\'].values():\n        rel = Path(asset[\'path\'])\n        path = (root / rel).resolve()\n        if rel.is_absolute() or not path.is_relative_to(root):\n            raise ValueError(\'Unsafe asset path\')\n        if not path.is_file() or path.stat().st_size != asset[\'bytes\']:\n            raise ValueError(f\'Missing/truncated asset: {rel}\')\n        if check_hash and sha256(path) != asset[\'sha256\']:\n            raise ValueError(f\'Checksum mismatch: {rel}\')\n    return record\n\ndef completed(root, config, check_hash=True):\n    rows = []\n    for spec in config[\'prompts\']:\n        p = Path(root) / \'metadata\' / f"{spec[\'id\']}.json"\n        if p.exists():\n            rows.append(verify_record(root, read_json(p), config, check_hash))\n    return rows\n\ndef refresh(root, config):\n    root = Path(root)\n    rows = completed(root, config)\n    tmp = root / \'manifest.jsonl.tmp\'\n    tmp.write_text(\'\'.join(json.dumps(r) + \'\\n\' for r in rows), encoding=\'utf-8\')\n    os.replace(tmp, root / \'manifest.jsonl\')\n    review = root / \'review.csv\'\n    old = {}\n    if review.exists():\n        with review.open(newline=\'\', encoding=\'utf-8\') as f:\n            old = {r[\'id\']: r for r in csv.DictReader(f)}\n    fields = [\'id\', \'keep\', \'action_complete\', \'object_consistent\', \'reverse_clear\', \'reason\']\n    tmp = root / \'review.csv.tmp\'\n    with tmp.open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=fields)\n        writer.writeheader()\n        for row in rows:\n            writer.writerow(old.get(row[\'id\'], dict(id=row[\'id\'], keep=\'\', action_complete=\'\', object_consistent=\'\', reverse_clear=\'\', reason=\'UNREVIEWED\')))\n    os.replace(tmp, review)\n    cards = []\n    for r in rows:\n        cards.append(f"<article><h2>{html.escape(r[\'id\'])} — {r[\'split\']}</h2><p>{html.escape(r[\'prompt\'])}</p><p><b>Reverse:</b> {html.escape(r[\'reverse_caption\'])}</p><p><b>Accept only if:</b> {html.escape(r[\'acceptance\'])}</p><video controls preload=\'none\' width=\'420\' src=\'{r[\'assets\'][\'forward\'][\'path\']}\'></video> <video controls preload=\'none\' width=\'420\' src=\'{r[\'assets\'][\'reverse\'][\'path\']}\'></video></article>")\n    (root / \'review.html\').write_text("<!doctype html><meta charset=\'utf-8\'><title>Teacher candidate review</title><h1>Forward and reversed teacher candidates</h1><p>Edit review.csv; blank keep means NOT accepted. A reversed clip is an inspection baseline.</p>" + \'\\n\'.join(cards), encoding=\'utf-8\')\n    return rows\n\ndef export_shard(root, destination):\n    root = Path(root)\n    config = read_json(root / \'dataset_config.json\')\n    rows = refresh(root, config)\n    if not rows:\n        raise ValueError(\'No completed candidates to export\')\n    files = {\'dataset_config.json\', \'manifest.jsonl\', \'review.csv\', \'review.html\'}\n    for r in rows:\n        files.add(f"metadata/{r[\'id\']}.json")\n        files.update(a[\'path\'] for a in r[\'assets\'].values())\n    checksums = {p: dict(sha256=sha256(root / p), bytes=(root / p).stat().st_size) for p in sorted(files)}\n    summary = dict(schema=SCHEMA, shard=config[\'shard\'], fingerprint=config[\'fingerprint\'], completed=len(rows), expected=10, files=checksums)\n    atomic_json(root / \'bundle_checksums.json\', summary)\n    files.add(\'bundle_checksums.json\')\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    tmp = destination.with_name(destination.name + \'.tmp\')\n    # NPZ and MP4 are already compressed. DEFLATE mainly shrinks padded embeddings.\n    with zipfile.ZipFile(tmp, \'w\', zipfile.ZIP_DEFLATED, compresslevel=1) as z:\n        for rel in sorted(files):\n            z.write(root / rel, f"shard_{config[\'shard\']:02d}/{rel}")\n    os.replace(tmp, destination)\n    return dict(path=str(destination), completed=len(rows), expected=10, bytes=destination.stat().st_size)\n')
(WORK/'wan_shards_worker.py').write_text('"""GPU worker. Each stage is a process; do not launch two writers for one shard."""\nimport argparse\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nfrom wan_shards_common import read_json, atomic_json, sha256, refresh, completed, verify_record\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'stage\', choices=[\'encode\', \'generate\', \'status\'])\n    p.add_argument(\'--root\', required=True)\n    p.add_argument(\'--limit\', type=int, default=1, help=\'Maximum newly completed clips in this call; <=0 means all remaining\')\n    args = p.parse_args()\n    root = Path(args.root)\n    c = read_json(root / \'dataset_config.json\')\n    if args.stage == \'status\':\n        rows = refresh(root, c)\n        print(f"Verified complete: {len(rows)}/10", flush=True)\n        return\n    import torch\n    from safetensors.torch import load_file, save_file\n    from diffusers import WanPipeline, AutoencoderKLWan, UniPCMultistepScheduler\n    assert torch.cuda.is_available(), \'A GPU is required. Select a T4 runtime.\'\n    print(\'GPU:\', torch.cuda.get_device_name(0), flush=True)\n    print(\'Dataset:\', c[\'fingerprint\'], flush=True)\n    print(\'Sampling:\', c[\'generation\'], flush=True)\n    done = {r[\'id\'] for r in completed(root, c)}\n    specs = [r for r in c[\'prompts\'] if r[\'id\'] not in done]\n    if args.limit > 0:\n        specs = specs[:args.limit]\n    if not specs:\n        print(\'No new candidates requested; all verified completed clips are skipped.\', flush=True)\n        refresh(root, c)\n        return\n    kw = dict(revision=c[\'revision\'], low_cpu_mem_usage=True, use_safetensors=True)\n    embdir = root / \'embeddings\'\n    embdir.mkdir(exist_ok=True)\n    negative_path = embdir / \'negative.safetensors\'\n    def save_tensors(path, values):\n        path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n        tmp = path.with_name(path.name + \'.tmp\')\n        save_file({k: v.detach().cpu().contiguous() for k, v in values.items()}, str(tmp))\n        os.replace(tmp, path)\n    if args.stage == \'encode\':\n        pending = [r for r in specs if not (embdir / f"{r[\'id\']}.safetensors").exists()]\n        if not pending and negative_path.exists():\n            print(\'Reusing all needed prompt embeddings.\', flush=True)\n            return\n        from transformers import AutoTokenizer, UMT5EncoderModel, BitsAndBytesConfig\n        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)\n        print(\'Loading quantized UMT5. Model downloads may take time.\', flush=True)\n        encoder = UMT5EncoderModel.from_pretrained(c[\'model\'], subfolder=\'text_encoder\', quantization_config=quant, torch_dtype=torch.float32, device_map={\'\': 0}, **kw).eval()\n        encoder.requires_grad_(False)\n        tokenizer = AutoTokenizer.from_pretrained(c[\'model\'], subfolder=\'tokenizer\', revision=c[\'revision\'])\n        pipe = WanPipeline(tokenizer=tokenizer, text_encoder=encoder, vae=None, scheduler=None, transformer=None)\n        def encode(text):\n            with torch.inference_mode():\n                pos, _ = pipe.encode_prompt(prompt=text, do_classifier_free_guidance=False, max_sequence_length=512, device=torch.device(\'cuda\'), dtype=torch.float32)\n            if not torch.isfinite(pos).all():\n                raise RuntimeError(\'Non-finite UMT5 embedding\')\n            return pos\n        if not negative_path.exists():\n            negatives = {r[\'negative_prompt\'] for r in c[\'prompts\']}\n            assert len(negatives) == 1\n            save_tensors(negative_path, {\'negative_prompt_embeds\': encode(next(iter(negatives)))})\n        for r in pending:\n            print(\'Encoding forward and reverse captions:\', r[\'id\'], flush=True)\n            # The reverse caption is stored for future student conditioning, not used to generate teacher targets.\n            forward = encode(r[\'prompt\'])\n            reverse = encode(r[\'reverse_caption\'])\n            save_tensors(embdir / f"{r[\'id\']}.safetensors", {\'forward_prompt_embeds\': forward, \'reverse_prompt_embeds\': reverse})\n            del forward, reverse\n        print(\'Encoding complete. Process exit releases the text encoder.\', flush=True)\n        return\n    for r in specs:\n        assert (embdir / f"{r[\'id\']}.safetensors").exists(), f"Encode {r[\'id\']} first"\n    assert negative_path.exists()\n    print(\'Loading Wan transformer and FP32 VAE; text encoder is not loaded.\', flush=True)\n    load_start = time.perf_counter()\n    vae = AutoencoderKLWan.from_pretrained(c[\'model\'], subfolder=\'vae\', torch_dtype=torch.float32, **kw)\n    pipe = WanPipeline.from_pretrained(c[\'model\'], vae=vae, text_encoder=None, tokenizer=None, torch_dtype=torch.float16, **kw)\n    pipe.transformer.requires_grad_(False).eval()\n    pipe.vae.requires_grad_(False).eval()\n    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=c[\'generation\'][\'flow_shift\'])\n    pipe.vae.enable_tiling()\n    pipe.enable_model_cpu_offload()\n    print(\'Model load seconds:\', round(time.perf_counter() - load_start, 1), flush=True)\n    negative = load_file(str(negative_path))[\'negative_prompt_embeds\']\n    import numpy as np\n    from diffusers.utils import export_to_video\n    g = c[\'generation\']\n    for r in specs:\n        print(\'\\nGenerating\', r[\'id\'], \'|\', r[\'split\'], \'| seed\', r[\'seed\'], flush=True)\n        print(r[\'prompt\'], flush=True)\n        torch.manual_seed(r[\'seed\'])\n        torch.cuda.reset_peak_memory_stats()\n        torch.cuda.synchronize()\n        begin = time.perf_counter()\n        captured = {}\n        def callback(pipeline, i, timestep, values):\n            lat = values[\'latents\']\n            if not torch.isfinite(lat).all():\n                raise RuntimeError(\'Non-finite denoising latents; stop and diagnose numerical precision before accepting this sample.\')\n            if i + 1 == g[\'steps\']:\n                captured[\'forward_latent\'] = lat.detach().cpu().contiguous()\n            if (i + 1) % 5 == 0 or i == 0:\n                print(f"{r[\'id\']} step {i+1}/{g[\'steps\']}; elapsed {(time.perf_counter()-begin)/60:.1f} min", flush=True)\n            return values\n        embpath = embdir / f"{r[\'id\']}.safetensors"\n        emb = load_file(str(embpath))\n        with torch.inference_mode():\n            frames = pipe(prompt_embeds=emb[\'forward_prompt_embeds\'].to(\'cuda\', torch.float16), negative_prompt_embeds=negative.to(\'cuda\', torch.float16), height=g[\'height\'], width=g[\'width\'], num_frames=g[\'frames\'], num_inference_steps=g[\'steps\'], guidance_scale=g[\'guidance\'], generator=torch.Generator(\'cpu\').manual_seed(r[\'seed\']), callback_on_step_end=callback, output_type=\'np\').frames[0]\n        torch.cuda.synchronize()\n        seconds = time.perf_counter() - begin\n        peak = torch.cuda.max_memory_allocated() / 2**30\n        if frames.shape != (g[\'frames\'], g[\'height\'], g[\'width\'], 3) or not np.isfinite(frames).all():\n            raise RuntimeError(\'Unexpected/non-finite decoded frames\')\n        uint8_frames = np.rint(np.clip(frames, 0, 1) * 255).astype(np.uint8)\n        paths = dict(forward=f"forward/{r[\'id\']}.mp4", reverse=f"reverse/{r[\'id\']}.mp4", frames_uint8=f"frames/{r[\'id\']}.npz", forward_latent=f"latents/{r[\'id\']}.safetensors", embedding=f"embeddings/{r[\'id\']}.safetensors", negative_embedding=\'embeddings/negative.safetensors\')\n        for rel in paths.values():\n            (root / rel).parent.mkdir(parents=True, exist_ok=True)\n        for key, pixels in [(\'forward\', uint8_frames), (\'reverse\', uint8_frames[::-1].copy())]:\n            dest = root / paths[key]\n            tmp = dest.with_name(dest.stem + \'.partial.mp4\')\n            export_to_video(pixels.astype(np.float32) / 255.0, str(tmp), fps=g[\'fps\'])\n            os.replace(tmp, dest)\n        dest = root / paths[\'frames_uint8\']\n        tmp = dest.with_name(dest.name + \'.tmp\')\n        with tmp.open(\'wb\') as f:\n            np.savez_compressed(f, frames=uint8_frames)\n        os.replace(tmp, dest)\n        save_tensors(root / paths[\'forward_latent\'], captured)\n        assets = {k: dict(path=v, sha256=sha256(root / v), bytes=(root / v).stat().st_size) for k, v in paths.items()}\n        record = dict(**r, schema=c[\'schema\'], status=\'completed_candidate\', model=c[\'model\'], revision=c[\'revision\'], generation=g, dataset_fingerprint=c[\'fingerprint\'], assets=assets, generation_and_decode_seconds=seconds, peak_gpu_gib=peak, gpu=torch.cuda.get_device_name(0), raw_frames_layout=\'THWC uint8\', forward_latent_note=\'Normalized forward Wan diffusion latent; do not flip this tensor to construct reverse targets. Reverse decoded frames and re-encode with Wan VAE.\', review_status=\'unreviewed\', text_encoder_note=\'Frozen UMT5, NF4 4-bit weights and FP32 compute; both captions encoded separately.\')\n        # This JSON is the commit marker. Unfinished candidates have no metadata record and are regenerated.\n        atomic_json(root / \'metadata\' / f"{r[\'id\']}.json", record)\n        refresh(root, c)\n        print(f"SAVED {r[\'id\']} | {seconds/60:.1f} min | {peak:.2f} GiB | not yet quality-approved", flush=True)\n        del frames, uint8_frames, captured, emb\n        torch.cuda.empty_cache()\n    print(\'This batch is complete. Export and review before training.\', flush=True)\n\nif __name__ == \'__main__\':\n    main()\n')
(WORK/'import_wan_shards.py').write_text('"""Import transfer ZIPs into a separate Wan dataset namespace. No ML packages required."""\nimport argparse\nimport csv\nimport html\nimport json\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nfrom wan_shards_common import (SCHEMA, MODEL, REVISION, GENERATION, atomic_json,\n                               read_json, sha256, initialize, completed, refresh, shard_plan)\n\n\ndef checked_extract(bundle, destination):\n    with zipfile.ZipFile(bundle) as z:\n        if sum(i.file_size for i in z.infolist()) > 12 * 2**30:\n            raise ValueError(\'Archive exceeds 12 GiB extraction limit\')\n        seen = set()\n        tops = set()\n        for info in z.infolist():\n            p = PurePosixPath(info.filename)\n            if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in info.filename or \':\' in info.filename:\n                raise ValueError(\'Unsafe archive path\')\n            if info.filename in seen or stat.S_ISLNK(info.external_attr >> 16):\n                raise ValueError(\'Duplicate entry or symlink in archive\')\n            seen.add(info.filename)\n            if p.parts:\n                tops.add(p.parts[0])\n        if len(tops) != 1 or next(iter(tops)) not in {f\'shard_{i:02d}\' for i in range(1, 5)}:\n            raise ValueError(\'Expected exactly one numbered shard per archive\')\n        z.extractall(destination)\n    return Path(destination) / next(iter(tops))\n\n\ndef verify_bundle(source, plan):\n    summary = read_json(source / \'bundle_checksums.json\')\n    if summary[\'schema\'] != SCHEMA:\n        raise ValueError(\'Unknown archive schema\')\n    for rel, expected in summary[\'files\'].items():\n        path = (source / rel).resolve()\n        if not path.is_relative_to(source.resolve()) or not path.is_file():\n            raise ValueError(\'Invalid inventory path\')\n        if path.stat().st_size != expected[\'bytes\'] or sha256(path) != expected[\'sha256\']:\n            raise ValueError(f\'Archive integrity failure: {rel}\')\n    c = read_json(source / \'dataset_config.json\')\n    if c[\'shard\'] != summary[\'shard\'] or source.name != f"shard_{c[\'shard\']:02d}":\n        raise ValueError(\'Shard identity mismatch\')\n    if c[\'model\'] != MODEL or c[\'revision\'] != REVISION or c[\'generation\'] != GENERATION:\n        raise ValueError(\'Incompatible teacher or sampling configuration\')\n    # Rebuild exact expected config from the distributed master plan.\n    expected_rows = shard_plan(plan, c[\'shard\'])\n    if c[\'prompts\'] != expected_rows or len(expected_rows) != 10:\n        raise ValueError(\'Prompt plan mismatch\')\n    from wan_shards_common import digest\n    without = {k: v for k, v in c.items() if k != \'fingerprint\'}\n    if digest(without) != c[\'fingerprint\'] or summary[\'fingerprint\'] != c[\'fingerprint\']:\n        raise ValueError(\'Config fingerprint mismatch\')\n    rows = completed(source, c)\n    manifest = [json.loads(s) for s in (source / \'manifest.jsonl\').read_text().splitlines() if s.strip()]\n    if manifest != rows or len(rows) != summary[\'completed\']:\n        raise ValueError(\'Manifest/completed-record mismatch\')\n    expected_files = {\'dataset_config.json\', \'manifest.jsonl\', \'review.csv\', \'review.html\'}\n    for r in rows:\n        expected_files.add(f"metadata/{r[\'id\']}.json")\n        expected_files.update(a[\'path\'] for a in r[\'assets\'].values())\n    if set(summary[\'files\']) != expected_files:\n        raise ValueError(\'Incomplete or unexpected inventory\')\n    return c, rows\n\n\ndef copy_immutable(source, destination):\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.exists():\n        if sha256(source) != sha256(destination):\n            raise ValueError(f\'Existing file conflicts; nothing is overwritten: {destination}\')\n        return\n    tmp = destination.with_name(destination.name + \'.importing\')\n    shutil.copy2(source, tmp)\n    tmp.replace(destination)\n\n\ndef import_bundle(bundle, target, plan, replace_review=False):\n    with tempfile.TemporaryDirectory() as d:\n        source = checked_extract(bundle, d)\n        config, rows = verify_bundle(source, plan)\n        destination = Path(target) / \'shards\' / source.name\n        initialize(destination, config[\'shard\'], plan)\n        # Verify all existing immutable paths before starting copies.\n        paths = {\'dataset_config.json\'}\n        for r in rows:\n            paths.add(f"metadata/{r[\'id\']}.json")\n            paths.update(a[\'path\'] for a in r[\'assets\'].values())\n        for rel in paths:\n            dest = destination / rel\n            if dest.exists() and sha256(source / rel) != sha256(dest):\n                raise ValueError(f\'Conflicting immutable artifact: {dest}\')\n        for rel in sorted(paths):\n            copy_immutable(source / rel, destination / rel)\n        local_review = destination / \'review.csv\'\n        previous = {}\n        if local_review.exists():\n            with local_review.open(newline=\'\', encoding=\'utf-8\') as f:\n                previous = {r[\'id\']: r for r in csv.DictReader(f)}\n        with (source / \'review.csv\').open(newline=\'\', encoding=\'utf-8\') as f:\n            incoming = list(csv.DictReader(f))\n        valid_ids = {r[\'id\'] for r in rows}\n        if len({r[\'id\'] for r in incoming}) != len(incoming) or {r[\'id\'] for r in incoming} != valid_ids:\n            raise ValueError(\'Review IDs do not match completed candidates\')\n        for r in incoming:\n            if replace_review or r[\'id\'] not in previous:\n                previous[r[\'id\']] = r\n        fields = [\'id\', \'keep\', \'action_complete\', \'object_consistent\', \'reverse_clear\', \'reason\']\n        with local_review.open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n            w = csv.DictWriter(f, fieldnames=fields); w.writeheader()\n            w.writerows(previous.values())\n        refresh(destination, config)\n        return len(rows)\n\n\ndef rebuild(target, plan):\n    target = Path(target)\n    merged, accepted, cards = [], [], []\n    expected = {r[\'id\']: r for r in plan}\n    yes = lambda value: str(value).strip().lower() in (\'1\', \'true\', \'yes\')\n    for shard in range(1, 5):\n        root = target / \'shards\' / f\'shard_{shard:02d}\'\n        if not (root / \'dataset_config.json\').exists():\n            continue\n        c = read_json(root / \'dataset_config.json\')\n        initialize(root, shard, plan)\n        rows = refresh(root, c)\n        with (root / \'review.csv\').open(newline=\'\', encoding=\'utf-8\') as f:\n            decisions = {r[\'id\']: r for r in csv.DictReader(f)}\n        for row in rows:\n            if row[\'id\'] not in expected or any(row[k] != expected[row[\'id\']][k] for k in expected[row[\'id\']]):\n                raise ValueError(\'Unexpected candidate labels\')\n            r = dict(row)\n            r[\'assets\'] = {k: dict(a, path=f"shards/shard_{shard:02d}/{a[\'path\']}") for k, a in row[\'assets\'].items()}\n            decision = decisions.get(r[\'id\'], {})\n            r[\'review\'] = decision\n            passes = all(yes(decision.get(k, \'\')) for k in (\'keep\', \'action_complete\', \'object_consistent\', \'reverse_clear\'))\n            r[\'review_status\'] = \'accepted\' if passes else \'not_accepted\'\n            merged.append(r)\n            if passes:\n                accepted.append(r)\n            review_rel = f\'shards/shard_{shard:02d}/review.csv\'\n            cards.append(f"<article><h2>{html.escape(r[\'id\'])} — {r[\'split\']} — {r[\'review_status\']}</h2><p>{html.escape(r[\'prompt\'])}</p><p>Accept only if: {html.escape(r[\'acceptance\'])}</p><video controls preload=\'none\' width=\'420\' src=\'{r[\'assets\'][\'forward\'][\'path\']}\'></video> <video controls preload=\'none\' width=\'420\' src=\'{r[\'assets\'][\'reverse\'][\'path\']}\'></video><p>Edit {review_rel}</p></article>")\n    order = {r[\'id\']: i for i, r in enumerate(plan)}\n    if len({r[\'id\'] for r in merged}) != len(merged):\n        raise ValueError(\'Duplicate candidates\')\n    for filename, rows in [(\'manifest_wan.jsonl\', merged), (\'accepted_manifest_wan.jsonl\', accepted)]:\n        rows.sort(key=lambda r: order[r[\'id\']])\n        tmp = target / (filename + \'.tmp\')\n        tmp.write_text(\'\'.join(json.dumps(r) + \'\\n\' for r in rows), encoding=\'utf-8\')\n        tmp.replace(target / filename)\n    summary = dict(schema=SCHEMA, model=MODEL, revision=REVISION, generation=GENERATION,\n                   expected_candidates=40, completed_candidates=len(merged), accepted_candidates=len(accepted),\n                   completed_splits={s: sum(r[\'split\'] == s for r in merged) for s in (\'train\', \'val\', \'test\')},\n                   accepted_splits={s: sum(r[\'split\'] == s for r in accepted) for s in (\'train\', \'val\', \'test\')},\n                   training_compatibility=\'Wan-specific dataset. Existing Zeroscope cache/train commands are NOT compatible.\')\n    atomic_json(target / \'dataset_summary.json\', summary)\n    (target / \'review_all.html\').write_text("<!doctype html><meta charset=\'utf-8\'><title>Wan candidate review</title><h1>Wan dataset candidates</h1><p>Edit each shard review.csv, then run import_wan_shards.py --refresh-only.</p>" + \'\\n\'.join(cards), encoding=\'utf-8\')\n    return summary\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--project\', required=True, help=\'Path to reverse_video_project\')\n    p.add_argument(\'--bundles\', nargs=\'*\', default=[])\n    p.add_argument(\'--plan\', default=str(Path(__file__).resolve().parent.parent / \'prompts_40.json\'))\n    p.add_argument(\'--refresh-only\', action=\'store_true\')\n    p.add_argument(\'--replace-review\', action=\'store_true\', help=\'Explicitly replace local review decisions with incoming ones\')\n    p.add_argument(\'--require-complete\', action=\'store_true\')\n    args = p.parse_args()\n    if not args.bundles and not args.refresh_only:\n        p.error(\'Supply --bundles or --refresh-only\')\n    plan = read_json(args.plan)\n    target = Path(args.project) / \'data\' / \'wan_pilot_v1\'\n    target.mkdir(parents=True, exist_ok=True)\n    for bundle in args.bundles:\n        print(\'Importing:\', bundle)\n        import_bundle(bundle, target, plan, args.replace_review)\n    summary = rebuild(target, plan)\n    print(json.dumps(summary, indent=2))\n    print(\'Dataset:\', target)\n    if args.require_complete and summary[\'completed_candidates\'] != 40:\n        raise SystemExit(\'Import is partial: fewer than 40 completed candidates. Existing imports remain saved.\')\n\nif __name__ == \'__main__\':\n    main()\n')
# Helpers use only standard Python, so importing them in the notebook kernel is safe.
import importlib.util
spec=importlib.util.spec_from_file_location('wan_shards_common',WORK/'wan_shards_common.py')
common=importlib.util.module_from_spec(spec); spec.loader.exec_module(common)
config=common.initialize(ROOT,SHARD,PLAN)
print('Frozen model revision:',config['revision'])
for i,r in enumerate(config['prompts'],1):
    print(f"{i:02d}. {r['id']} | {r['event']} | {r['split']} | seed {r['seed']}")
    print(r['prompt'])
    print('Reverse caption:',r['reverse_caption'])
    print('Accept only if:',r['acceptance'],'\n')


## 4. Define resumable stages and check existing results
A clip is skipped only if its metadata and required files/checksums verify. Corruption stops the run rather than silently skipping or spending GPU time on a replacement. Partial files with no committed metadata can be regenerated. No parallel writers for one shard.

In [ ]:
RUNNER=WORK/'wan_shards_worker.py'
def stage(name,limit=1):
    stream([PY,'-u',RUNNER,name,'--root',ROOT,'--limit',str(limit)])
stage('status')


## 5. Encode the first unfinished candidate
The frozen quantized text encoder runs in its own process. It encodes the forward and reverse captions separately, then exits. Teacher video generation uses the **forward** caption. Reverse targets are created by reversing the resulting frames, not by asking the teacher for a reverse action.

In [ ]:
stage('encode',limit=1)


## 6. Generate one candidate at the full 50-step settings
This can take about 45 minutes on the previously measured T4. Progress prints every five steps. It does not resume within an unfinished video. Stop and share logs on errors rather than reinstalling arbitrary packages.

In [ ]:
stage('generate',limit=1)


## 7. Inspect the first available sample
Check the acceptance condition, then watch both directions. A sharp but motionless or incomplete action is not usable. For falling, merely disappearing below the table is insufficient: the floor contact must be visible.

In [ ]:
from IPython.display import display,Video
records=common.refresh(ROOT,config)
assert records, 'No completed candidates.'
row=records[0]
print(row['id'],row['split'])
print(row['prompt'])
print('Accept only if:',row['acceptance'])
display(Video(str(ROOT/row['assets']['forward']['path']),embed=True))
print('Reverse teacher — inspection baseline:')
display(Video(str(ROOT/row['assets']['reverse']['path']),embed=True))
print('Generation + decode minutes:',round(row['generation_and_decode_seconds']/60,1))


## 8. Generate the remaining candidates only after inspecting the sample
Set the approval variable to True after checking the action and deciding the remaining work is worthwhile. This is a local spending/quality check, not a model-quality guarantee. `NEW_CLIPS_THIS_RUN=9` permits the rest of the ten. Lower it to 1 or 2 if you want smaller batches. Encoding and generation select the same next unfinished candidates. Rerun this cell to continue after completed batches; rerun setup after a lost runtime.

In [ ]:
APPROVE_REMAINING=False  # Set True after inspecting the first generated sample.
NEW_CLIPS_THIS_RUN=9
if APPROVE_REMAINING:
    stage('encode',limit=NEW_CLIPS_THIS_RUN)
    stage('generate',limit=NEW_CLIPS_THIS_RUN)
else:
    print('Remaining batch paused. Inspect the sample before setting APPROVE_REMAINING=True.')


## 9. Review every completed candidate
This creates an offline HTML gallery and an editable CSV. Nothing is accepted automatically. Set **keep=1, action_complete=1, object_consistent=1, reverse_clear=1** only for usable clips, and explain any rejection. Reject malformed/ambiguous events. Do not change labels to match a failed generation. The 32/4/4 split is fixed in the prompt plan and must not be reassigned after seeing quality.

In [ ]:
records=common.refresh(ROOT,config)
print('Complete:',len(records),'/10')
print('Review CSV:',ROOT/'review.csv')
print('Offline gallery:',ROOT/'review.html')
for r in records:
    print(r['id'],r['split'],'|',r['acceptance'])
# To watch another completed candidate, change this index and rerun:
REVIEW_INDEX=0
if records:
    r=records[REVIEW_INDEX]
    print('Viewing:',r['id'])
    display(Video(str(ROOT/r['assets']['forward']['path']),embed=True))
    display(Video(str(ROOT/r['assets']['reverse']['path']),embed=True))


## 10. Export this shard for transfer
Run even for a partial shard before ending a session; export includes only verified completed candidates. Re-export after finishing or editing review.csv. The ZIP contains both videos, raw uint8 frames, embeddings, latent, captions, metadata, fixed split, and checksums. It excludes model weights. ZIP creation can take several minutes. A partial export does not mean ten videos are complete.

In [ ]:
BUNDLE=EXPORTS/f'wan_pilot_v1_shard_{SHARD:02d}.zip'
summary=common.export_shard(ROOT,BUNDLE)
print(json.dumps(summary,indent=2))
print('Transfer this ZIP to the central project machine or Drive. Do not copy only the MP4s.')
# Optional browser download:
# from google.colab import files
# files.download(str(BUNDLE))


## 11. Import the four bundles centrally
The download package includes a standard-Python importer. On the central PC, extract the package and run from its folder:

```bash
python tools/import_wan_shards.py --project /path/to/reverse_video_project --bundles /path/to/wan_pilot_v1_shard_01.zip /path/to/wan_pilot_v1_shard_02.zip /path/to/wan_pilot_v1_shard_03.zip /path/to/wan_pilot_v1_shard_04.zip --require-complete
```

On Windows, quote paths containing spaces. Python 3.10+ is required for this small import tool; no GPU or PyTorch is needed. Shards may be imported one at a time by passing one ZIP and omitting `--require-complete`.

Destination: **reverse_video_project/data/wan_pilot_v1/**. Open `review_all.html`, edit the per-shard `review.csv`, then run:

```bash
python tools/import_wan_shards.py --project /path/to/reverse_video_project --refresh-only
```

Use `accepted_manifest_wan.jsonl` only after review. Blank review decisions are excluded. Existing local review decisions are preserved on reimport; `--replace-review` explicitly replaces them from the incoming bundles.

**Training boundary:** the earlier project's Zeroscope cache, LoRA, loss and train modules are not Wan compatible. These notebooks generate/import a well-labeled Wan dataset; they do not implement or validate Wan student training. Reverse decoded frames, then re-encode with Wan's temporal VAE; reversing forward latent time is not equivalent.

Validation: prompt counts, split assignments, notebook syntax, checksums, resume/merge and review rules are tested with synthetic files. The newly detailed prompts and full four-shard CUDA runs have not been executed by the author. Neither all-candidate quality nor training fit is guaranteed.

References: [official model](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B-Diffusers), [pinned pipeline](https://huggingface.co/docs/diffusers/v0.35.1/api/pipelines/wan), [Colab resource policy](https://research.google.com/colaboratory/faq.html#usage-limits).
